In [ ]:
import numpy as np
import pandas as pd
import gymnasium as gym
from gymnasium import spaces
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.callbacks import CheckpointCallback, EvalCallback
from stable_baselines3.common.logger import Logger
from stable_baselines3.common.monitor import Monitor


In [ ]:
# Класс для модели котла
class Boiler:
    def __init__(self):
        self.power = 0

    def set_power(self, power):
        self.power = power

    def get_heat_gain(self):
        return self.power * 1000 * 0.99  # КПД обогревателя 99%

# Класс для модели кондиционера
class AirConditioner:
    def __init__(self):
        self.status = 0

    def set_status(self, status):
        self.status = status

    def get_heat_gain(self, current_temp, setpoint):
        if self.status == 1:
            if current_temp > setpoint:
                return -3 * 1000  # Охлаждение
            else:
                return 3 * 1000  # Нагрев
        return 0

# Модель нагрева
class HomeTempModel:
    def __init__(self, outside_data):
        self.outside_data = outside_data
        self.length = 10  # м
        self.width = 10  # м
        self.height = 3  # м
        self.wall_thickness = 0.4  # м (пенобетон)
        self.insulation_thickness = 0.05  # м (ПСБС)
        self.roof_thickness = 0.15  # м (ПСБС)
        self.floor_thickness = 0.15  # м (бетон)
        self.floor_insulation_thickness = 0.1  # м (ПСБС)
        self.wall_conductivity = 0.14  # Вт/(м·К) (пенобетон)
        self.insulation_conductivity = 0.05  # Вт/(м·К) (ПСБС)
        self.roof_conductivity = 0.05  # Вт/(м·К) (ПСБС)
        self.floor_conductivity = 1.5  # Вт/(м·К) (бетон)
        self.setpoint = 23.0  # Уставка температуры
        # Телоемкость внутреннего объема дома.
        # Подобрал "от балды", чтобы было похоже на правду.
        # Если считать просто по объему воздуха, будет очень быстрый нагрев/охлаждение
        self.inner_heat_capacity = 20
        self.boiler = Boiler()
        self.air_conditioner = AirConditioner()

    def get_inside_temp(self, current_temp, current_time, step_interval):
        outside_temp = self.outside_data.get_parameter('temperature', current_time)
        attic_temp = outside_temp * 0.9
        basement_temp = current_temp * 0.7
        wall_area = 2 * (self.length * self.height + self.width * self.height)
        roof_area = self.length * self.width
        floor_area = self.length * self.width
        total_conductivity_wall = (self.wall_thickness / self.wall_conductivity + self.insulation_thickness / self.insulation_conductivity)
        total_conductivity_roof = (self.roof_thickness / self.roof_conductivity)
        total_conductivity_floor = (self.floor_thickness / self.floor_conductivity + self.floor_insulation_thickness / self.insulation_conductivity)
        heat_loss_walls = wall_area * (current_temp - outside_temp) / total_conductivity_wall
        heat_loss_roof = roof_area * (current_temp - attic_temp) / total_conductivity_roof
        heat_loss_floor = floor_area * (current_temp - basement_temp) / total_conductivity_floor
        total_heat_loss = heat_loss_walls + heat_loss_roof + heat_loss_floor

        heat_gain = self.boiler.get_heat_gain() + self.air_conditioner.get_heat_gain(current_temp, self.setpoint)

        temp_change = (heat_gain - total_heat_loss) * step_interval * 60 / (self.length * self.width * self.height * self.inner_heat_capacity * 1000)
        new_temp = current_temp + temp_change
        return new_temp

    def get_tariff(self, current_time):
        if current_time.hour >= 7 and current_time.hour < 23:
            if current_time.month in [1, 2, 3, 4, 5, 6]:
                return 3.06
            else:
                return 3.20
        else:
            if current_time.month in [1, 2, 3, 4, 5, 6]:
                return 1.15
            else:
                return 1.20

# Среда (RL environment)
class HouseEnv(gym.Env):
    def __init__(self, outside_data, step_interval=20, setpoint=23, hysteresis=2, render_mode=None):
        super(HouseEnv, self).__init__()
        # Пространство действий:
        # Обогреватель : 0:выключен, 1: 3 кВт, 2: 6 кВт, 3: 9 кВт, 4: кондиционер
        self.action_space = spaces.Discrete(5)
        #Пространство состояний:
        # 1. Текущая температура в доме (-50:50) - self.current_temp
        # 2. Текущая температура снаружи  (-50:50) - self.outside_temp
        # 3. Уставка (10:30) - self.setpoint
        # 4. Гистерезис (0:5) - self.hysteresis
        # 5. Месяц (1:12) - self.day
        # 6. День (1:31) - self.month
        # 7. Текущий тариф - день/ночь (0:10) - self.current_tariff
        # 8. Состояние кондиционера - ошибка/включился (0/1) - self.ac_status
        self.observation_space = spaces.Box(low=np.array([-50, -50, 10, 0, 1, 1, 0, 0]), high=np.array([50, 50, 30, 5, 12, 31, 10, 1]), dtype=np.float32)
        self.setpoint = setpoint
        self.hysteresis = hysteresis
        self.step_interval = step_interval
        self.home_temp_model = HomeTempModel(outside_data)
        self.max_steps = 0  # Максимальное количество шагов в эпизоде (будет рассчитано при reset)
        self.current_step = 0
        self.render_mode = render_mode
        self.episode_reward = 0  # Суммарная награда за эпизод
        self.temp_diff_reward = 0
        self.power_reward = 0
        self.last_action = None  # Последнее действие
        self.total_steps = 0

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.current_temp = self.setpoint #Начинаем без отклонений
        #Определяем время начала эпизода. Или заданный месяц, или случайный
        if (options is None): #Год / месяц не задан - определяем случайный.
            self.year = np.random.randint(2006, 2024)  # Случайный год из диапазона 2006-2023
            self.month = np.random.randint(1, 13)  # Случайный месяц
        else:
            self.year = options['year']
            self.month = options['month']
        self.day = 1
        self.current_time = pd.Timestamp(year=self.year, month=self.month, day=self.day, hour=0, minute=0)
        self.current_tariff = self.home_temp_model.get_tariff(self.current_time)
        self.current_step = 0
        self.max_steps = (self.current_time + pd.DateOffset(months=1) - self.current_time).total_seconds() \
                        // (self.step_interval * 60) #Количество шагов эпизода
        self.episode_reward = 0  # Сброс суммарной награды за эпизод
        self.home_temp_model.air_conditioner.set_status(0)  # Сброс статуса кондиционера
        self.home_temp_model.boiler.set_power(0)  # Сброс статуса котла
        self.outside_temp = self.home_temp_model.outside_data.get_parameter('temperature', self.current_time)
        self.temp_good_for_ac = -5
        # Для логов
        self.temp_hit = 0
        self.temp_miss = 0
        self.power_cost = 0
        self.all_off = 0
        self.ac_heat = 0
        self.ac_cool = 0
        self.heater3 = 0
        self.heater6 = 0
        self.heater9 = 0
        self.last_action = None  # Сброс последнего действия

        print(f"------------------------------------------------------------------------")
        print(f"Selected month for the episode: {self.month}.{self.year}")  # Вывод выбранного месяца и года
        return np.array([self.current_temp,
                         self.outside_temp,
                         self.setpoint,
                         self.hysteresis,
                         self.month,
                         self.day,
                         self.current_tariff,
                         self.home_temp_model.air_conditioner.status]), {}

    def step(self, action):
        if action == 4:  # Кондиционер
            if self.outside_temp > self.temp_good_for_ac:
                self.home_temp_model.air_conditioner.set_status(1)
                self.home_temp_model.boiler.set_power(1.1)
            else:
                self.home_temp_model.air_conditioner.set_status(0)
                self.home_temp_model.boiler.set_power(0)
        else:
            self.home_temp_model.air_conditioner.set_status(0)
            self.home_temp_model.boiler.set_power([0, 3, 6, 9][action])

        cost = self.home_temp_model.boiler.power * self.current_tariff * (self.step_interval / 60)  # Стоимость электроэнергии за шаг
        prev_temp=self.current_temp
        self.current_temp = self.home_temp_model.get_inside_temp(self.current_temp, self.current_time, self.step_interval)
        self.current_time += pd.Timedelta(minutes=self.step_interval)
        self.current_tariff = self.home_temp_model.get_tariff(self.current_time)
        self.outside_temp = self.home_temp_model.outside_data.get_parameter('temperature', self.current_time)

        # reward = self.hysteresis-1.5*abs(self.current_temp - self.setpoint)*abs(self.current_temp - self.setpoint)
        if abs(self.current_temp - self.setpoint) <= 0.9*self.hysteresis:
            reward = 1
        elif abs(self.current_temp - self.setpoint) <= self.hysteresis:
            reward = 0
        else:
            reward = -10

        if abs(self.current_temp - self.setpoint) <= self.hysteresis:
        #     reward = 2*(1-abs(self.current_temp - self.setpoint)/self.hysteresis)  # Штраф за отклонения в пределах гистерезиса
             self.temp_hit += 1
        else:
        #     reward = -0.5*abs(self.current_temp - self.setpoint)  # Штраф за отклонения вне пределов гистерезиса
            self.temp_miss += 1
        # if self.outside_temp < prev_temp and prev_temp > self.setpoint and action == 4:
        #     reward -= 10
        if abs(prev_temp - self.setpoint) > abs (self.current_temp - self.setpoint):
            reward += 0.05 #Награда за движение в правильном направлении
        self.power_cost += cost
        reward -= 0.6*cost  # Штраф за расход электроэнергии (0.5)
        if self.home_temp_model.boiler.power == 0:
            self.all_off += 1
        if self.home_temp_model.boiler.power == 1.1 and prev_temp > self.setpoint:
            self.ac_cool += 1
        if self.home_temp_model.boiler.power == 1.1 and prev_temp < self.setpoint:
            self.ac_heat += 1
        if self.home_temp_model.boiler.power == 3:
            self.heater3 += 1
        if self.home_temp_model.boiler.power == 6:
            self.heater6 += 1
        if self.home_temp_model.boiler.power == 9:
            self.heater9 += 1

        self.episode_reward += reward  # Добавление награды к суммарной награде за эпизод
        done = self.current_step >= self.max_steps
        self.current_step += 1
        self.total_steps += 1
        self.last_action = action  # Обновление последнего действия

        if done:
            print(f"Episode finished. Ep. reward: {self.episode_reward:.0f}. Total steps:{self.total_steps}")  # Вывод суммарной награды за эпизод
            print(f"Temp hit/miss:     {self.temp_hit*self.step_interval / 60:.0f}/{self.temp_miss*self.step_interval / 60:.0f}")
            print(f"All off:           {self.all_off*self.step_interval / 60:.0f}")
            print(f"Heater 3/6/9:      {self.heater3*self.step_interval / 60:.0f}/{self.heater6*self.step_interval / 60:.0f}/{self.heater9*self.step_interval / 60:.0f}")
            print(f"AC heat/cool:      {self.ac_heat*self.step_interval / 60:.0f}/{self.ac_cool*self.step_interval / 60:.0f}")
            print(f"Power cost:        {self.power_cost:.0f} RUR")
            print(f"------------------------------------------------------------------------")
        return np.array([self.current_temp,
                         self.outside_temp,
                         self.setpoint,
                         self.hysteresis,
                         self.month,
                         self.day,
                         self.current_tariff,
                         self.home_temp_model.air_conditioner.status]), reward, done, False, {'ep_reward':self.episode_reward,
                        'hit':self.temp_hit*self.step_interval / 60,
                        'miss':self.temp_miss
                        }
    def render(self, mode='human'):
        if self.render_mode is None:
            gym.logger.warn(
                "You are calling render method without specifying any render mode."
            )
            return
        if self.render_mode == 'human':
            print(f"Date: {self.current_time.strftime('%d.%m.%Y %H:%M')}, Current Temp: {self.current_temp:.2f}, Outside Temp: {self.home_temp_model.outside_data.get_parameter('temperature', self.current_time):.2f}, Setpoint: {self.setpoint:.2f}, Tariff: {self.current_tariff:.2f}, Boiler Power:{self.home_temp_model.boiler.power} kW, AC Status: {self.home_temp_model.air_conditioner.status}")
        else:
            gym.logger.warn(
                "This environment supports only 'human' render mode"
            )

    def simple_action(self, obs):
        #Простой алгоритм управления сравнения с нейронкой

        current_temp = obs[0]
        outside_temp = obs[1]
        setpoint = obs[2]
        hysteresis = obs[3]

        # Дома хорошо, ничего не включаем
        if abs(current_temp - setpoint) <= hysteresis*0.25:
            action = 0
        # Если отклонение от уставки > 25% величины гистерезиса и позволяют условия, запускаем кондиционер (неважно, нагрев или охлаждение, он сам разберется)
        if abs(current_temp - setpoint) > hysteresis*0.25 and outside_temp >= self.temp_good_for_ac:
            action = 4
        # Температура на улице не годится для кондиционера, и дома немного холоднее чем надо - включаем котел на 3 кВт
        if outside_temp < self.temp_good_for_ac and (setpoint - current_temp) > self.hysteresis*0.25:
            action = 1
        # Дома прилично холоднее чем надо - включаем котел на 6 кВт
        if outside_temp < self.temp_good_for_ac and (setpoint - current_temp) > self.hysteresis*0.5:
            action = 2
        # Хьюстон, у нас проблемы, жена мерзнет, - включаем котел на 9 кВт
        if outside_temp < self.temp_good_for_ac and (setpoint - current_temp) > self.hysteresis*0.75:
            action = 3
        return action